In [ ]:

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
 accuracy_score, precision_recall_fscore_support,
 roc_auc_score, confusion_matrix, classification_report
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.calibration import CalibratedClassifierCV
import numpy as np
import os
import json
ember_base_dir =
"/content/ember_dataset/ember_dataset_2018_2/ember_dataset_2018_2/ember2018"
def read_ember_jsonl(filepath):
 X_list, y_list = [], []
 with open(filepath, "r") as f:
 for line in f:
 data = json.loads(line)
 label = data.get("label")
 if label == -1:
 continue
 features = []
 features.extend(data.get("histogram", []))
 features.extend(data.get("byteentropy", []))
 strings = data.get("strings", {})
 features += [
 strings.get("numstrings", 0),
 strings.get("avlength", 0.0),
 strings.get("printables", 0),
 strings.get("entropy", 0.0),
 strings.get("paths", 0),
 strings.get("urls", 0),
 strings.get("registry", 0),
 strings.get("MZ", 0),
 ]
 printabledist = strings.get("printabledist", [0]*96)
 features.extend(printabledist[:96] + [0]*(96-len(printabledist)))
 general = data.get("general", {})
 features += [
 general.get("size", 0),
 general.get("vsize", 0),
 general.get("has_debug", 0),
 general.get("exports", 0),
 general.get("imports", 0),
 general.get("has_relocations", 0),
 general.get("has_resources", 0),
 general.get("has_signature", 0),
 general.get("has_tls", 0),
 general.get("symbols", 0),
 ]
 X_list.append(features)
 y_list.append(label)
 max_len = max(len(x) for x in X_list)
 X = np.array([x + [0]*(max_len-len(x)) for x in X_list], dtype=np.float32)
 y = np.array(y_list, dtype=np.int32)
 return X, y
# Load data
X_train_list, y_train_list = [], []
for i in range(6):
 Xp, yp = read_ember_jsonl(os.path.join(ember_base_dir, f"train_features_{i}.jsonl"))
 X_train_list.append(Xp)
 y_train_list.append(yp)
X_train = np.vstack(X_train_list)
y_train = np.hstack(y_train_list)
X_test, y_test = read_ember_jsonl(os.path.join(ember_base_dir, "test_features.jsonl"))
# CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
dt_train_oof_probs = np.zeros(len(X_train), dtype=np.float32)
dt_test_probs = np.zeros(len(X_test), dtype=np.float32)
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
 print(f"\nFold {fold}")
 X_tr, X_val = X_train[train_idx], X_train[val_idx]
 y_tr, y_val = y_train[train_idx], y_train[val_idx]
 dt_base = DecisionTreeClassifier(
 max_depth=10,
 min_samples_leaf=200,
 class_weight="balanced",
 random_state=42
 )
 model = CalibratedClassifierCV(dt_base, method="isotonic", cv=3)
 model.fit(X_tr, y_tr)
 val_probs = model.predict_proba(X_val)[:, 1]
 dt_train_oof_probs[val_idx] = val_probs
 y_val_pred = (val_probs >= 0.5).astype(int)
 print("ROC-AUC:", roc_auc_score(y_val, val_probs))
 dt_test_probs += model.predict_proba(X_test)[:, 1]
dt_test_probs /= 5
np.save("dt_train_oof_probs.npy", dt_train_oof_probs)
np.save("dt_train_labels.npy", y_train)
np.save("dt_test_probs.npy", dt_test_probs)
np.save("dt_test_labels.npy", y_test)
print("✅ DT stacking files generated correctly")
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
accuracy_score,
precision_recall_fscore_support,
roc_auc_score,
confusion_matrix,
classification_report
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.calibration import CalibratedClassifierCV
import numpy as np
import os
import json
# EMBER base directory
ember_base_dir =
'/content/ember_dataset/ember_dataset_2018_2/ember_dataset_2018_2/ember2018'
def read_ember_jsonl(filepath):
X_list, y_list = [], []
with open(filepath, 'r') as f:
for line in f:
data = json.loads(line)
label = data.get('label')
if label == -1:
continue
features = []
features.extend(data.get('histogram', []))
features.extend(data.get('byteentropy', []))
strings = data.get('strings', {})
features += [
strings.get('numstrings', 0),
strings.get('avlength', 0.0),
strings.get('printables', 0),
strings.get('entropy', 0.0),
strings.get('paths', 0),
strings.get('urls', 0),
strings.get('registry', 0),
strings.get('MZ', 0),
]
printabledist = strings.get('printabledist', [0] * 96)
features.extend(printabledist[:96] + [0] * (96 - len(printabledist)))
general = data.get('general', {})
features += [
general.get('size', 0),
general.get('vsize', 0),
general.get('has_debug', 0),
general.get('exports', 0),
general.get('imports', 0),
general.get('has_relocations', 0),
general.get('has_resources', 0),
general.get('has_signature', 0),
general.get('has_tls', 0),
general.get('symbols', 0),
]
X_list.append(features)
y_list.append(label)
max_len = max(len(x) for x in X_list)
X = np.array([x + [0] * (max_len - len(x)) for x in X_list], dtype=np.float32)
y = np.array(y_list, dtype=np.int32)
return X, y
# Load training data
X_train_list, y_train_list = [], []
for i in range(6):
Xp, yp = read_ember_jsonl(os.path.join(ember_base_dir, f'train_features_{i}.jsonl'))
X_train_list.append(Xp)
y_train_list.append(yp)
X_train = np.vstack(X_train_list)
y_train = np.hstack(y_train_list)
# Load test data
X_test, y_test = read_ember_jsonl(os.path.join(ember_base_dir, 'test_features.jsonl'))
# CV setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
dt_test_probs = np.zeros(len(X_test), dtype=np.float32)
# Train folds ONLY to generate final test probabilities
for train_idx, val_idx in skf.split(X_train, y_train):
X_train_fold = X_train[train_idx]
y_train_fold = y_train[train_idx]
dt_base = DecisionTreeClassifier(
max_depth=10,
min_samples_leaf=200,
class_weight="balanced",
random_state=42
)
model = CalibratedClassifierCV(dt_base, method="isotonic", cv=3)
model.fit(X_train_fold, y_train_fold)
dt_test_probs += model.predict_proba(X_test)[:, 1]
dt_test_probs /= 5
# Final test predictions
y_test_pred = (dt_test_probs >= 0.5).astype(int)
# ===== FINAL TEST OUTPUTS =====
accuracy = accuracy_score(y_test, y_test_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
y_test, y_test_pred, average='binary', zero_division=0
)
roc_auc = roc_auc_score(y_test, dt_test_probs)
print("\n===== FINAL TEST RESULTS (Decision Tree) =====")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall : {recall:.4f}")
print(f"F1-Score : {f1:.4f}")
print(f"ROC-AUC : {roc_auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred, zero_division=0))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))
Given that the EMBER dataset is quite large, we applied the algorithm and then attempted
Stratified K - Fold Cross Validation to determine if there is any improvement with cross
validation. Below are the average results with k = 5 on the train dataset.
Metrics Values
Accuracy
Precision
Recall
F1-score
ROC-AUC
89.74%
 65 0.8765
0.9252
0.9002
0.9622
Here is the final result on test dataset :
Metrics Values
Accuracy
Precision
Recall
F1-score
ROC-AUC
0.8781
0.8402
0.9337
0.8845
0.9539
Confusion Metrix
Predicted Benign (0) Predicted Malware (1)
Actual Benign (0) 82245 17755
Actual Malware (1) 6631 93369